# Artifact Results Overview

This notebook reads the saved evaluation outputs in `src/artifacts` and summarizes the image, text, and multimodal model results.

In [ ]:
from collections import Counter
from html import escape
from pathlib import Path
import csv
import json

from IPython.display import HTML, Image, display

artifact_root = Path("src/artifacts")
if not artifact_root.exists():
    artifact_root = Path("artifacts")
if not artifact_root.exists():
    raise FileNotFoundError("Could not find an artifacts directory.")

artifact_dirs = sorted(path for path in artifact_root.iterdir() if path.is_dir())
model_names = [path.name for path in artifact_dirs]

TABLE_STYLE = """
<style>
table.results-table {
    border-collapse: collapse;
    margin: 0.75rem 0 1.25rem;
}
table.results-table th,
table.results-table td {
    border: 1px solid #d0d7de;
    padding: 0.35rem 0.6rem;
    text-align: left;
}
table.results-table th {
    background: #f6f8fa;
}
</style>
"""


def format_value(value):
    if isinstance(value, float):
        return f"{value:.4f}"
    return str(value)


def rows_to_html(rows, columns=None):
    if not rows:
        return "<p>No rows to display.</p>"
    if columns is None:
        columns = list(rows[0].keys())

    header = "".join(f"<th>{escape(str(column))}</th>" for column in columns)
    body = []

    for row in rows:
        cells = "".join(
            f"<td>{escape(format_value(row.get(column, '')))}</td>" for column in columns
        )
        body.append(f"<tr>{cells}</tr>")

    return (
        '<table class="results-table">'
        + f"<thead><tr>{header}</tr></thead>"
        + f"<tbody>{''.join(body)}</tbody>"
        + "</table>"
    )


display(HTML(TABLE_STYLE))
model_names

: 

In [ ]:
metrics_by_model = {}
summary_rows = []

for artifact_dir in artifact_dirs:
    metrics_path = artifact_dir / "test" / "metrics.json"
    with metrics_path.open() as handle:
        metrics = json.load(handle)

    metrics_by_model[artifact_dir.name] = metrics
    summary_rows.append(
        {
            "model": artifact_dir.name,
            "accuracy": metrics["acc"],
            "macro_f1": metrics["macro_f1"],
            "weighted_f1": metrics["weighted_f1"],
            "macro_roc_auc": metrics["macro_roc_auc"],
            "loss": metrics["loss"],
            "misclassified_total": int(metrics["misclassified_total"]),
            "samples_per_sec": metrics["samples_per_sec"],
        }
    )

summary_rows = sorted(summary_rows, key=lambda row: row["accuracy"], reverse=True)
display(HTML(rows_to_html(summary_rows)))

In [ ]:
class_names = next(iter(metrics_by_model.values()))["class_names"]
per_class_f1_rows = []

for label in class_names:
    row = {"label": label}
    for model_name in model_names:
        row[model_name] = metrics_by_model[model_name]["per_class"][label]["f1"]
    per_class_f1_rows.append(row)

display(HTML(rows_to_html(per_class_f1_rows, columns=["label", *model_names])))

In [ ]:
for artifact_dir in artifact_dirs:
    display(HTML(f"<h3>{escape(artifact_dir.name)}</h3>"))
    display(HTML("<p><strong>Confusion matrix</strong></p>"))
    display(Image(filename=str(artifact_dir / "test" / "confusion_matrix.png")))
    display(HTML("<p><strong>ROC curves</strong></p>"))
    display(Image(filename=str(artifact_dir / "test" / "roc_curves.png")))

In [ ]:
misclassified_by_model = {}
top_error_pair_rows = []
top_confident_rows = []

for artifact_dir in artifact_dirs:
    with (artifact_dir / "test" / "misclassified.csv").open(newline="") as handle:
        rows = list(csv.DictReader(handle))

    for row in rows:
        row["confidence"] = float(row["confidence"])
        row["filename"] = Path(row["path"]).name

    misclassified_by_model[artifact_dir.name] = rows

    error_pairs = Counter((row["true_label"], row["pred_label"]) for row in rows)
    for (true_label, pred_label), count in error_pairs.most_common(5):
        top_error_pair_rows.append(
            {
                "model": artifact_dir.name,
                "true_label": true_label,
                "pred_label": pred_label,
                "count": count,
            }
        )

    for row in sorted(rows, key=lambda current: current["confidence"], reverse=True)[:10]:
        top_confident_rows.append(
            {
                "model": artifact_dir.name,
                "filename": row["filename"],
                "true_label": row["true_label"],
                "pred_label": row["pred_label"],
                "confidence": row["confidence"],
                "raw_text": row["raw_text"],
            }
        )

display(HTML("<h3>Most common error pairs</h3>"))
display(HTML(rows_to_html(top_error_pair_rows)))

display(HTML("<h3>Highest-confidence mistakes</h3>"))
display(
    HTML(
        rows_to_html(
            top_confident_rows,
            columns=["model", "filename", "true_label", "pred_label", "confidence", "raw_text"],
        )
    )
)